# 04 · Anchor-free detector: FCOS

## Setup

In [ ]:
# Setup: clone/update the repo on Colab/Kaggle, install requirements, resolve paths, get the data
import os, sys, subprocess
REPO_URL = "https://github.com/anna-kryvulia/ucu-ai-hw-object-detection.git"
BRANCH = "main"   # branch with the code you pushed
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    ROOT = "/content/repo" if "google.colab" in sys.modules else "/kaggle/working/repo"
    if not os.path.exists(ROOT):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/requirements.txt"], check=True)
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, f"{ROOT}/src")

from hwdet import env
from hwdet.data.download import ensure_data
from hwdet.tracker import Run, status
P = env.setup(seed=42)
ensure_data(P, labels_only=False)   # True on the Mac: annotations only
status(P)                           # which runs / stages are already done

In [ ]:
# Imports, plot style, W&B login (the key is read from Kaggle / Colab secrets; without it W&B is skipped)
import math
import re
import time
import types
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchvision
from torch import nn

from hwdet import CLASS_NAMES, NUM_CLASSES
from hwdet.config import load_config
from hwdet.data.detection_dataset import DetectionDataset, build_targets
from hwdet.data.io import frame_table
from hwdet.engine import Data, run_experiment
from hwdet.env import REPO_ROOT
from hwdet.eval.coco_eval import evaluate, evaluate_by_size
from hwdet.tracker import load_registry
from hwdet.viz import INK_3, PALETTE, save_fig, set_style
from hwdet.wandb_utils import login_from_secrets

NOTEBOOK_START = time.time()            # the tuned run is skipped if too little of Kaggle's 12 h is left
set_style()
login_from_secrets()
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device", env.device())

## 1. Data
Same protocol as Faster R-CNN and RetinaNet (`configs/experiments/_protocol.yaml`): every 10th training
frame, the full official val split for final metrics, every 5th val frame after each epoch, 8 epochs at
batch size 4.

In [ ]:
# Frames and targets (COCO annotations -> torchvision boxes/labels + ignored regions), built once
protocol = load_config("fcos_baseline")["split"]
train_frames = frame_table(P.data, "train")
train_frames = train_frames[(train_frames.frame - 1) % protocol["train_frame_step"] == 0]
val_frames = frame_table(P.data, "val")
monitor_frames = val_frames[(val_frames.frame - 1) % protocol["monitor_frame_step"] == 0]
test_frames = frame_table(P.data, "test")
targets = build_targets(P.data, pd.concat([train_frames, val_frames]))
data = Data(P.data, train_frames, val_frames, monitor_frames, targets, test_frames)
pd.DataFrame({"frames": [len(train_frames), len(val_frames), len(monitor_frames), len(test_frames)],
              "clips": [d["clip"].nunique() for d in (train_frames, val_frames, monitor_frames, test_frames)]},
             index=["train (subsampled)", "val", "val monitor", "test"])

## 2. Model
torchvision FCOS (ResNet-50 + FPN, GroupNorm heads), pretrained on COCO. There are no anchor boxes: every
location of every FPN level (P3–P7, strides 8–128) predicts class scores, the distances from the location
to the four sides of the box (l, t, r, b) and a centerness score. At inference the score is the geometric
mean of the class probability and the centerness, which down-ranks boxes predicted far from an object's
centre.

The COCO classification layer is replaced with a new one for 9 outputs (index 0 unused, so labels stay 1–8
as for the other detectors), initialised with the prior probability 0.01. torchvision fixes the focal loss
(α = 0.25, γ = 2) and the GIoU box loss, so the head loss is replaced by the same function with both read
from the config.

**P2 level (`p2_head`).** The FPN also takes the stride-4 ResNet stage (layer1), giving levels P2–P7. All
COCO weights are loaded (the FPN layers move up one index); only the new P2 lateral and output convolutions
start from their default initialisation. The heads are shared across levels, so they need no new weights.

In [ ]:
# Builds FCOS from a config: COCO weights, optional P2 level, new 9-output classifier, configurable head loss
from torchvision.models import resnet50
from torchvision.models.detection import FCOS, FCOS_ResNet50_FPN_Weights, fcos_resnet50_fpn
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.backbone_utils import _resnet_fpn_extractor
from torchvision.ops import complete_box_iou_loss, distance_box_iou_loss, generalized_box_iou_loss, sigmoid_focal_loss
from torchvision.ops.feature_pyramid_network import LastLevelP6P7
from torchvision.ops.misc import FrozenBatchNorm2d

MODEL_KWARGS = ["min_size", "max_size", "score_thresh", "nms_thresh", "detections_per_img",
                "topk_candidates", "center_sampling_radius"]
BOX_LOSSES = {"giou": generalized_box_iou_loss, "diou": distance_box_iou_loss, "ciou": complete_box_iou_loss}


def fcos_with_p2(**kwargs):
    """FCOS on P2-P7. A location's "anchor size" equals its stride; torchvision derives the centre-sampling
    radius and the size range of each level from it, so P2 gets radius 1.5 * 4 px and range 0-32 px."""
    backbone = _resnet_fpn_extractor(resnet50(norm_layer=FrozenBatchNorm2d), 3, returned_layers=[1, 2, 3, 4],
                                     extra_blocks=LastLevelP6P7(256, 256))
    sizes = ((4,), (8,), (16,), (32,), (64,), (128,))
    model = FCOS(backbone, 91, anchor_generator=AnchorGenerator(sizes, ((1.0,),) * len(sizes)), **kwargs)
    state = FCOS_ResNet50_FPN_Weights.COCO_V1.get_state_dict(progress=False)
    shift = lambda k: re.sub(r"fpn\.(inner|layer)_blocks\.(\d)\.", lambda g: f"fpn.{g[1]}_blocks.{int(g[2]) + 1}.", k)
    missing, unexpected = model.load_state_dict({shift(k): v for k, v in state.items()}, strict=False)
    assert not unexpected and all("_blocks.0." in k for k in missing), (missing, unexpected)
    return model


def fcos_head_loss(self, targets, head_outputs, anchors, matched_idxs):
    """torchvision's FCOS head loss, with the focal loss parameters and the box loss taken from the head."""
    cls_logits, bbox_regression = head_outputs["cls_logits"], head_outputs["bbox_regression"]
    gt_classes, gt_boxes = [], []
    for tgt, matched in zip(targets, matched_idxs):
        if len(tgt["labels"]) == 0:
            cls_t, box_t = tgt["labels"].new_zeros((len(matched),)), tgt["boxes"].new_zeros((len(matched), 4))
        else:
            cls_t, box_t = tgt["labels"][matched.clip(min=0)], tgt["boxes"][matched.clip(min=0)]
        cls_t[matched < 0] = -1                                       # background location
        gt_classes.append(cls_t)
        gt_boxes.append(box_t)
    gt_classes, gt_boxes, anchors = torch.stack(gt_classes), torch.stack(gt_boxes), torch.stack(anchors)
    fg = gt_classes >= 0
    n_fg = max(1, fg.sum().item())

    onehot = torch.zeros_like(cls_logits)
    onehot[fg, gt_classes[fg]] = 1.0
    loss_cls = sigmoid_focal_loss(cls_logits, onehot, alpha=self.focal_alpha, gamma=self.focal_gamma, reduction="sum")

    pred_boxes = self.box_coder.decode(bbox_regression.float(), anchors)   # float: IoU losses are unstable in fp16
    loss_box = self.box_loss(pred_boxes[fg], gt_boxes[fg], reduction="sum")

    reg_t = self.box_coder.encode(anchors, gt_boxes)                       # l, t, r, b of every location
    lr, tb = reg_t[..., [0, 2]], reg_t[..., [1, 3]]
    ctr_t = torch.sqrt((lr.min(-1).values / lr.max(-1).values) * (tb.min(-1).values / tb.max(-1).values))
    pred_ctr = head_outputs["bbox_ctrness"].squeeze(dim=2)
    loss_ctr = nn.functional.binary_cross_entropy_with_logits(pred_ctr[fg], ctr_t[fg], reduction="sum")
    return {"classification": loss_cls / n_fg, "bbox_regression": loss_box / n_fg, "bbox_ctrness": loss_ctr / n_fg}


def build_fcos(cfg):
    m = cfg["model"]
    kwargs = {k: m[k] for k in MODEL_KWARGS if k in m}
    if m["p2_head"]:
        model = fcos_with_p2(**kwargs)
    else:
        model = fcos_resnet50_fpn(weights=FCOS_ResNet50_FPN_Weights.COCO_V1, progress=False, **kwargs)
    head = model.head
    cls_head = head.classification_head
    cls_head.cls_logits = nn.Conv2d(256, NUM_CLASSES + 1, kernel_size=3, padding=1)
    nn.init.normal_(cls_head.cls_logits.weight, std=0.01)
    nn.init.constant_(cls_head.cls_logits.bias, -math.log((1 - 0.01) / 0.01))
    cls_head.num_classes = NUM_CLASSES + 1
    head.focal_alpha, head.focal_gamma, head.box_loss = m["focal_alpha"], m["focal_gamma"], BOX_LOSSES[m["box_loss"]]
    head.compute_loss = types.MethodType(fcos_head_loss, head)
    return model

## 3. How locations are assigned to objects
A location (the centre of a feature-map cell) is a positive for an object only if all of these hold
(torchvision `FCOS.compute_loss`):

1. it lies **inside** the box;
2. it is within **radius × stride** of the box centre in x and y (centre sampling, radius 1.5);
3. the largest of its four distances to the box sides falls into the **size range of its level**:
   P3 0–64 px, P4 64–128, P5 128–256, P6 256–512, P7 > 512 (4× to 8× the stride; with P2: P2 0–32, P3 32–64, …);
4. if several objects qualify, the location goes to the **smallest** one.

All other locations are negatives for the classifier; boxes and centerness are trained on positives only.
For a location near the centre that distance is about half the object side, so P3 takes objects up to
~128 px: almost every VisDrone object. An object about the size of the stride (8 px) or smaller covers
at most one or two locations, possibly none (condition 1). In
dense scenes neighbouring boxes overlap, and the shared locations go to the smaller object, so a
partly occluded larger object can lose most of them.

The counts below replay this rule (the model's own `compute_loss`) on 150 training frames, with the
training targets (boxes in ignored regions dropped) after the model's resize, for the default setting, the larger centre radius and the P2 level.

In [ ]:
# Positive locations per object, by object size and by scene density, from the model's own assignment rule
SIZE_BINS, SIZE_LABELS = [0, 16, 32, 96, 1e5], ["< 16 px", "16–32 px", "32–96 px", "> 96 px"]
DENSITY_BINS, DENSITY_LABELS = [-1, 20, 60, 1000], ["≤ 20 objects", "21–60", "> 60"]


def feature_sizes(h, w, strides):
    sizes = []
    for s in strides:                                                 # padded input is divisible by 32
        sizes.append((h // s, w // s) if s <= 32 else (math.ceil(sizes[-1][0] / 2), math.ceil(sizes[-1][1] / 2)))
    return sizes


def assign(model, img, tgt):
    """Resized boxes, location centres, level of each location and the matched object (-1 = negative)."""
    il, t = model.transform([img], [{"boxes": tgt["boxes"], "labels": tgt["labels"]}])
    strides = [int(a[0]) for a in model.anchor_generator.sizes]       # sizes = ((8,), (16,), ...) = strides
    shapes = feature_sizes(*il.tensors.shape[-2:], strides)
    feats = [torch.empty(1, 1, h, w) for h, w in shapes]
    anchors = model.anchor_generator(il, feats)[0]
    matched = model.compute_loss(t, None, [anchors], [h * w for h, w in shapes])[0]
    level = torch.cat([torch.full((h * w,), k) for k, (h, w) in enumerate(shapes)])
    return il, t[0]["boxes"], (anchors[:, :2] + anchors[:, 2:]) / 2, level, matched


def assignment_model(config):
    model = build_fcos(load_config(config)).eval()
    model.head.compute_loss = lambda targets, head_outputs, anchors, matched_idxs: matched_idxs   # matching only
    return model


def assignment_stats(config, frames, n=150):
    model = assignment_model(config)
    ds = DetectionDataset(frames, P.data, targets, train=True, hflip=0.0)   # training targets, no flip
    rows = []
    for i in np.linspace(0, len(ds) - 1, n).astype(int):
        img, tgt, _ = ds[i]
        if len(tgt["boxes"]) == 0:
            continue
        _, boxes, _, level, matched = assign(model, img, tgt)
        pos = matched >= 0
        count = torch.bincount(matched[pos], minlength=len(boxes))
        main = [int(level[pos][matched[pos] == j].mode().values) if count[j] else -1 for j in range(len(boxes))]
        wh = boxes[:, 2:] - boxes[:, :2]
        rows.append(pd.DataFrame({"side": wh.prod(1).sqrt().numpy(), "positives": count.numpy(),
                                  "level": main, "objects_in_frame": len(boxes)}))
    objs = pd.concat(rows, ignore_index=True)
    objs["size"] = pd.cut(objs.side, SIZE_BINS, labels=SIZE_LABELS)
    objs["density"] = pd.cut(objs.objects_in_frame, DENSITY_BINS, labels=DENSITY_LABELS)
    return objs


ASSIGN_CONFIGS = ["fcos_baseline", "fcos_center", "fcos_p2"]
assign_stats = {c: assignment_stats(c, train_frames) for c in ASSIGN_CONFIGS}
no_pos = pd.DataFrame({c: o.groupby("size", observed=False).positives.apply(lambda p: (p == 0).mean())
                       for c, o in assign_stats.items()}).T
no_pos.insert(0, "all objects", [(o.positives == 0).mean() for o in assign_stats.values()])
for d in DENSITY_LABELS:
    no_pos[d] = [(o[o.density == d].positives == 0).mean() for o in assign_stats.values()]
print("objects without any positive location, % (by size after resizing, then by objects per frame)")
display((no_pos * 100).round(1))
print("positive locations per object (median)")
pd.DataFrame({c: o.groupby("size", observed=False).positives.median() for c, o in assign_stats.items()}).T

In [ ]:
# FPN level that holds each object's positives (share of objects, %)
level_names = {"fcos_baseline": ["P3", "P4", "P5", "P6", "P7"], "fcos_p2": ["P2", "P3", "P4", "P5", "P6", "P7"]}
pd.DataFrame({c: o[o.level >= 0].level.map(dict(enumerate(level_names[c]))).value_counts(normalize=True) * 100
              for c, o in assign_stats.items() if c in level_names}).T.reindex(columns=level_names["fcos_p2"]).round(1)

In [ ]:
# Positive locations in the densest 200 x 200 px window of a crowded training frame: P3-P7 vs P2-P7
def densest_frame(frames):
    n = frames.file_name.map(lambda f: len(targets[f]["boxes"]))
    return frames.loc[n.idxmax()]


ds = DetectionDataset(train_frames, P.data, targets, train=True, hflip=0.0)
row = densest_frame(train_frames)
img, tgt, _ = ds[train_frames.index.get_loc(row.name)]
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, config in zip(axes, ["fcos_baseline", "fcos_p2"]):
    il, boxes, centres, level, matched = assign(assignment_model(config), img, tgt)
    h, w = il.image_sizes[0]
    shown = nn.functional.interpolate(img[None], size=(h, w), mode="bilinear")[0].clamp(0, 1).permute(1, 2, 0).numpy()
    c = ((boxes[:, :2] + boxes[:, 2:]) / 2).numpy()
    grid = [(x, y) for x in range(0, w - 200, 20) for y in range(0, h - 200, 20)]
    x0, y0 = max(grid, key=lambda p: ((c[:, 0] >= p[0]) & (c[:, 0] < p[0] + 200) & (c[:, 1] >= p[1]) & (c[:, 1] < p[1] + 200)).sum())
    ax.imshow(shown[y0:y0 + 200, x0:x0 + 200], extent=(x0, x0 + 200, y0 + 200, y0))
    for j, (bx0, by0, bx1, by1) in enumerate(boxes.numpy()):
        ax.add_patch(plt.Rectangle((bx0, by0), bx1 - bx0, by1 - by0, fill=False, lw=1.2, color=PALETTE[j % len(PALETTE)]))
    inside = (centres[:, 0] >= x0) & (centres[:, 0] < x0 + 200) & (centres[:, 1] >= y0) & (centres[:, 1] < y0 + 200)
    neg, pos = inside & (matched < 0), inside & (matched >= 0)
    ax.scatter(centres[neg, 0], centres[neg, 1], s=3, color="white", alpha=0.5)
    ax.scatter(centres[pos, 0], centres[pos, 1], s=14, c=[PALETTE[j % len(PALETTE)] for j in matched[pos].tolist()],
               edgecolors="black", linewidths=0.4)
    ax.set(xlim=(x0, x0 + 200), ylim=(y0 + 200, y0), xticks=[], yticks=[],
           title=f"{config}: positive locations (box colour), negatives (white)")
fig.tight_layout()
save_fig(fig, "fcos_assignment", P)

## 4. Smoke test
One short epoch on a few frames for each model variant (default, P2 level, CIoU loss) under separate run
ids, to catch errors and out-of-memory problems before the long runs. They are deleted afterwards.
Metrics after 10 steps are close to 0: the new classification layer has not been trained yet.

In [ ]:
# Smoke tests: 1 epoch on ~40 frames per variant, then the run folders and their registry rows are removed
import shutil
smoke = Data(P.data, train_frames.iloc[::60], val_frames.iloc[::150], val_frames.iloc[::150], targets)
smoke_ids = []
for config in ["fcos_baseline", "fcos_p2", "fcos_ciou"]:
    smoke_ids.append(f"smoke_{config}")
    run_experiment(config, build_fcos, P, smoke, run_id=smoke_ids[-1], wandb=False,
                   train={"epochs": 1, "warmup_iters": 5})
    shutil.rmtree(P.runs / smoke_ids[-1], ignore_errors=True)
reg_file = P.outputs / "experiments.csv"
if reg_file.exists():
    reg = pd.read_csv(reg_file)
    reg[~reg.run_id.isin(smoke_ids)].to_csv(reg_file, index=False)

## 5. Experiments
Each cell trains one config, evaluates the best epoch on the full val split and saves everything under
`outputs/runs/<config>/`. Re-running a cell after a crash continues where it stopped.

| config | parameter | search range | change vs baseline |
|---|---|---|---|
| `fcos_baseline` | – | – | torchvision defaults, lr 0.01, 300 detections per image |
| `fcos_center` | label assignment: centre-sampling radius | 1.5, 2.5 strides | 2.5 |
| `fcos_p2` | extra high-resolution level | P3–P7, P2–P7 | P2–P7 (objects up to ~64 px move to stride 4) |
| `fcos_ciou` | box regression loss | GIoU, CIoU | CIoU |

In [ ]:
# Baseline
runs = {}
runs["fcos_baseline"] = run_experiment("fcos_baseline", build_fcos, P, data)

In [ ]:
# Experiment 1: label assignment - centre-sampling radius 2.5 strides
runs["fcos_center"] = run_experiment("fcos_center", build_fcos, P, data)

In [ ]:
# Experiment 2: additional high-resolution level P2 (stride 4)
runs["fcos_p2"] = run_experiment("fcos_p2", build_fcos, P, data)

In [ ]:
# Experiment 3: CIoU box regression loss
runs["fcos_ciou"] = run_experiment("fcos_ciou", build_fcos, P, data)

## 6. Tuned model
The tuned model combines every change that raised val mAP by at least 0.005 on its own (smaller
differences are within run-to-run noise, see the Faster R-CNN runs). With exactly one such change the
tuned model is that run; with none it is the baseline. The tuned model also predicts the test split for
the submission.

Training the combination is skipped when less of Kaggle's 12-hour limit is left than it needs (3.5 h with
the P2 level, 1.5 h without). The notebook then finishes with the best single run; saving a new version
with this notebook's output attached as an input trains the combination and skips everything else.

In [ ]:
# Tuned FCOS: baseline + every change that helped; test predictions + submission for it
EXPERIMENTS = ["fcos_center", "fcos_p2", "fcos_ciou"]
val = load_registry(P).query("eval == 'val'").set_index("run_id")
gain = {r: val.loc[r, "mAP"] - val.loc["fcos_baseline", "mAP"] for r in EXPERIMENTS}
helped = [r for r in EXPERIMENTS if gain[r] >= 0.005]
print("mAP change vs baseline:", {r: round(g, 4) for r, g in gain.items()})

base_model = load_config("fcos_baseline")["model"]
changes = {k: v for r in helped for k, v in load_config(r)["model"].items() if base_model[k] != v}
hours_left = 12 - (time.time() - NOTEBOOK_START) / 3600
started = (P.runs / "fcos_tuned" / "history.csv").exists()
if len(helped) >= 2 and (started or hours_left >= (3.5 if changes.get("p2_head") else 1.5)):
    tuned_id = "fcos_tuned"
    run_experiment("fcos_baseline", build_fcos, P, data, predict_test=True, run_id=tuned_id, model=changes,
                   description="FCOS tuned: " + " + ".join(helped))
else:
    if len(helped) >= 2:
        print(f"only {hours_left:.1f} h left: fcos_tuned not trained, save a new version to train it")
    tuned_id = max(["fcos_baseline"] + helped, key=lambda r: val.loc[r, "mAP"])
    run_experiment(tuned_id, build_fcos, P, data, predict_test=True)
print("tuned model:", tuned_id)
final = load_config("fcos_baseline", model=changes) if tuned_id == "fcos_tuned" else load_config(tuned_id)
print("selected settings:", {k: final["model"][k] for k in ["p2_head", "center_sampling_radius", "box_loss"]})

## 7. Comparison with the tuned anchor-based detectors
The anchor-based references are the Faster R-CNN and RetinaNet runs with the best val mAP
(`results/experiments.csv`, from notebooks 02 and 03). All numbers are on the full val split.

In [ ]:
# Val metrics: tuned Faster R-CNN and RetinaNet, all FCOS runs (committed results + runs of this notebook)
reg = (pd.concat([pd.read_csv(REPO_ROOT / "results/experiments.csv"), load_registry(P)])
       .sort_values("updated").drop_duplicates(["run_id", "eval"], keep="last"))
val = reg[reg["eval"] == "val"].set_index("run_id")
frcnn_id = val[val.family == "two_stage"].mAP.idxmax()
retina_id = val[val.family == "one_stage"].mAP.idxmax()
order = [frcnn_id, retina_id, "fcos_baseline"] + EXPERIMENTS + (["fcos_tuned"] if tuned_id == "fcos_tuned" else [])
cols = ["description", "mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR_500", "ms_per_img"]
val.loc[order, cols].round(4)

In [ ]:
# Per-class AP (%)
(val.loc[order, [f"AP_{c}" for c in CLASS_NAMES]].rename(columns=lambda c: c[3:]) * 100).round(1)

In [ ]:
# AP by object size (COCO small / medium / large): tuned anchor-based detectors vs initial and tuned FCOS
models = list(dict.fromkeys([frcnn_id, retina_id, "fcos_baseline", tuned_id]))
metrics = ["AP_small", "AP_medium", "AP_large", "mAP"]
fig, ax = plt.subplots(figsize=(11, 3.8))
width = 0.8 / len(models)
for k, (r, col) in enumerate(zip(models, PALETTE)):
    xs = np.arange(len(metrics)) + (k - (len(models) - 1) / 2) * width
    ax.bar(xs, val.loc[r, metrics].astype(float), width * 0.92, color=col, label=r)
    for x, v in zip(xs, val.loc[r, metrics].astype(float)):
        ax.text(x, v + 0.005, f"{v:.3f}", ha="center", fontsize=7, color=INK_3)
ax.set_xticks(np.arange(len(metrics)), ["small", "medium", "large", "all (mAP)"])
ax.set(ylabel="AP (IoU 0.50:0.95)", title="Val AP by object size")
ax.legend(loc="upper left", frameon=False)
fig.tight_layout()
save_fig(fig, "fcos_vs_anchor_sizes", P)

**Finer size bins.** COCO's "small" (< 32 px) mixes tiny and small objects. The table splits it, using
object sides √(w·h) in original pixels and the saved val predictions. The Faster R-CNN and RetinaNet
predictions are available when the outputs of notebooks 02 and 03 are attached as inputs (restored by the
setup cell).

In [ ]:
# AP, AP50 and recall per object-size bin (original pixels) from saved val predictions
by_size = {}
for r in models:
    f = P.runs / r / "predictions" / "val.csv"
    if f.exists():
        by_size[r] = evaluate_by_size(pd.read_csv(f), val_frames, P.data)
    else:
        print(f"{r}: no val predictions in {P.runs}")
pd.concat(by_size, names=["run", "object side"]).round(4)

In [ ]:
# Training curves: val mAP (monitor subset) per epoch, FCOS runs vs the tuned anchor-based detectors
def history(run_id):
    f = P.runs / run_id / "history.csv"
    return pd.read_csv(f if f.exists() else REPO_ROOT / "results/runs" / run_id / "history.csv")


fig, (a0, a1) = plt.subplots(1, 2, figsize=(14, 4.2))
for r, col in zip(order, PALETTE):
    h = history(r)
    loss_cols = [c for c in h.columns if c.startswith("loss_") or c in ("classification", "bbox_regression", "bbox_ctrness")]
    ls = "--" if r in (frcnn_id, retina_id) else "-"
    a0.plot(h.epoch, h[loss_cols].sum(1), marker="o", color=col, ls=ls, label=r)
    a1.plot(h.epoch, h.val_mAP, marker="o", color=col, ls=ls, label=r)
a0.set(title="Training loss (sum of the model's losses)", xlabel="epoch")
a1.set(title="Val mAP (monitor subset)", xlabel="epoch")
a1.legend(loc="lower right")
fig.tight_layout()
save_fig(fig, "fcos_curves", P)

**Dense scenes.** Val frames are split by the number of annotated objects, and AP is computed per group
from the saved val predictions.

In [ ]:
# AP by scene density (objects per frame) from saved val predictions
n_obj = val_frames.file_name.map(lambda f: len(targets[f]["boxes"]))
density = pd.cut(n_obj, DENSITY_BINS, labels=DENSITY_LABELS)
rows = {}
for r in models:
    f = P.runs / r / "predictions" / "val.csv"
    if not f.exists():
        print(f"{r}: no val predictions in {P.runs}")
        continue
    preds = pd.read_csv(f)
    for d in density.cat.categories:
        fr = val_frames[density == d]
        m = evaluate(preds[preds.file_name.isin(fr.file_name)], fr, P.data, per_class=False)
        rows[(r, d)] = {"frames": len(fr), "mAP": m["mAP"], "AP50": m["AP50"], "AP_small": m["AP_small"]}
pd.DataFrame(rows).T.round(4)